# VGG-16 / Tiny-ImageNet-200 trên Kaggle T4 ×2 (80 epoch), chia 2 tài khoản
Thiết kế: paper_notes §24. Yêu cầu: Accelerator = **GPU T4 x2**, Internet = **On**, *Add Input* một Kaggle Dataset chứa thư mục `tiny-imagenet-200/` (có `wnids.txt`, `train/`, `val/`), ví dụ tìm "tiny-imagenet-200" trong Datasets.

| Tài khoản A | Tài khoản B |
|---|---|
| sweep lr (~0.5 h), `timeprest`, `variant1`, `bench_comm` (~0.25 h) | `pipedream`, `pipedream_vsync`, `variant2` |

Mỗi run ~2.5–3 h (xem `est_full_run_h` của D4); mỗi tài khoản ~8.5 h. Một session Kaggle tối đa 12 h và **/kaggle/working bị xoá khi session kết thúc**: sau mỗi run hãy chạy cell đóng gói rồi tải zip về. Session mới: chạy lại cell 1–4 (setup, dữ liệu, check) rồi tiếp các run còn lại.
Thứ tự: cả hai tài khoản chạy cell 1–4 → A chạy sweep, dán bảng cho agent → agent cập nhật lr → pull lại (cell 1) và chạy lại check (cell 4) → chạy các run.

## 1. Repo + cài đặt + môi trường (cả A và B)

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), '| nccl', torch.cuda.nccl.version())"
!find /kaggle/input -maxdepth 6 \( -name wnids.txt -o -name stats.json \)

## 2. Unit test (CPU) ~3 phút

In [ ]:
!python -m pytest -q

## 3. Dữ liệu: dùng cache đính kèm (dataset `tin-cache`, 5 file .npy/json) nếu có trong /kaggle/input; nếu không, giải mã `tiny-imagenet-200` vào `/kaggle/working/tin_cache` (~3–5 phút)

In [ ]:
!python -c "from timeprest.config import load_config; from timeprest.data import build_datasets; import json, os; c = load_config('configs/tin_base.yaml'); tr, te = build_datasets(c['data'], 200, 0); print('train', len(tr), 'val(test)', len(te), 'sample', tuple(tr[0][0].shape), '| cache', tr.cache_dir); print(json.load(open(os.path.join(tr.cache_dir, 'stats.json'))))"

## 4. Check D1–D4 trên Tiny-ImageNet (5 hệ) ~5 phút
Phải ra `ALL PHASE-2 CHECKS PASS: True`; ghi lại `est_full_run_h` của D4.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.checks --config configs/tin_quick.yaml

## 5. [A] Sweep lr (1 GPU, ~30 phút; lr 0.02/0.01 × TiMePReSt, PipeDream)
Dán bảng cuối cho agent. Agent cập nhật lr trong `configs/tin_*.yaml`; sau đó chạy lại cell 1 (pull) và cell 4 (check) ở **cả hai** tài khoản.

In [ ]:
!python -m timeprest.sweep --config configs/tin_sweep.yaml

## 6. [A] Các run của tài khoản A

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_timeprest.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_variant1.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.bench_comm --config configs/tin_bench_comm.yaml
from IPython.display import Image
display(Image('/kaggle/working/tin_bench_comm/bench_comm.png'))

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_A.zip runs/tin_vgg16_*/metrics.csv runs/tin_vgg16_*/summary.json runs/tin_vgg16_*/config.json runs/tin_vgg16_*/op_trace_epoch1.json runs/tin_vgg16_*/checks.json tin_bench_comm/bench_comm.* tin_bench_comm/*/metrics.csv tin_bench_comm/*/config.json tin_sweep/summary.csv
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_A.zip'))

## 7. [B] Các run của tài khoản B

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_pipedream.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_pipedream_vsync.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_variant2.yaml --resume

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_B.zip runs/tin_vgg16_*/metrics.csv runs/tin_vgg16_*/summary.json runs/tin_vgg16_*/config.json runs/tin_vgg16_*/op_trace_epoch1.json runs/tin_vgg16_*/checks.json 
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_B.zip'))

## 8. Đợt 2: thêm seed + kiểm tra BatchNorm (paper_notes §24.4)
Cần cell 1, 3, 4 (setup, dữ liệu, check) trước. Cùng một seed thì TiMePReSt và PipeDream chạy trên **cùng tài khoản** để so được cả thời gian. Các run có `op_trace_every: 10` (lưu thứ tự op mỗi 10 epoch).

| Tài khoản A (~5.8 h) | Tài khoản B (~5.2 h) |
|---|---|
| `tin_timeprest_s1`, `tin_pipedream_s1`, `tin_bn_seq_n3`, `tin_bn_seq_n1` (1 GPU, ~35 phút cả hai) | `tin_timeprest_s2`, `tin_pipedream_s2` |

### [A] seed 1 + BatchNorm

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_timeprest_s1.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_pipedream_s1.yaml --resume

In [ ]:
!python -m timeprest.train --config configs/tin_bn_seq_n3.yaml
!python -m timeprest.train --config configs/tin_bn_seq_n1.yaml

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_seeds_A.zip runs/tin_*_s*/ runs/tin_bn_seq_*/metrics.csv runs/tin_bn_seq_*/summary.json runs/tin_bn_seq_*/config.json -x '*.pt' 
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_seeds_A.zip'))

### [B] seed 2

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_timeprest_s2.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_pipedream_s2.yaml --resume

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_seeds_B.zip runs/tin_*_s*/ runs/tin_bn_seq_*/metrics.csv runs/tin_bn_seq_*/summary.json runs/tin_bn_seq_*/config.json -x '*.pt' 
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_seeds_B.zip'))